# IFRS S1 / S2 — Disclosure Catalog Pipeline

**Fully automated. No hardcoded page ranges. No hardcoded paragraph lists.**

Pipeline steps:
1. Detect inputs — accept PDF or Markdown, auto-convert PDF → Markdown
2. Parse paragraph IDs + text dynamically from Markdown
3. Freeze `ifrs_paragraphs.json` — immutable source of truth
4. Verify paragraph coverage (dynamic — checks actual vs expected pattern)
5. Classify requirement paragraphs via SHALL rule-based filter
6. LLM structures requirements — reads extracted text only, cannot invent
7. Validate every row — source_quote must exist in extracted paragraph text
8. Finalize catalog — assign IDs, deduplicate, save CSV + JSON
9. Write validation_report.md
10. Embed into ChromaDB (run after catalog is stable)


In [1]:
# ── Cell 0 — Install dependencies ────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--break-system-packages", *pkgs
    ])

pip("pymupdf", "pymupdf4llm", "anthropic", "tqdm", "pandas")
print("Dependencies ready.")

Dependencies ready.


In [ ]:
# ── Cell 1 — Imports and paths ────────────────────────────────────────────────
import fitz                        # PyMuPDF
import pymupdf4llm                 # PDF → clean Markdown
import re
import json
import hashlib
import os
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
import anthropic

# ── Directory layout ─────────────────────────────────────────────────────────
# Place your PDF or Markdown files here. Any mix works.
# Naming convention: the standard name is derived from the filename.
# ifrs_s1.pdf  → IFRS_S1
# ifrs_s2.md   → IFRS_S2
# You can add more standards later; the pipeline is fully generic.

INPUT_DIR   = Path("standards/raw")        # drop PDFs or MDs here
OUT_DIR     = Path("standards/processed")
MD_DIR      = Path("standards/markdown")   # PDFs get converted here

for d in [INPUT_DIR, OUT_DIR, MD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARA_JSON    = OUT_DIR / "ifrs_paragraphs.json"
CATALOG_CSV  = OUT_DIR / "disclosures_catalog.csv"
CATALOG_JSON = OUT_DIR / "disclosures_catalog.json"
VAL_REPORT   = OUT_DIR / "validation_report.md"

# Standard name mapping  —  filename stem → canonical label
# Pattern: any file whose lowercase stem contains 's1' → IFRS_S1, etc.
# Extend this dict if you add more standards.
STANDARD_NAME_MAP = {
    "s1": "IFRS_S1",
    "s2": "IFRS_S2",
}

def stem_to_standard(stem: str) -> str:
    """Derive canonical standard name from filename stem."""
    stem_lower = stem.lower()
    for key, name in STANDARD_NAME_MAP.items():
        if key in stem_lower:
            return name
    return stem.upper()   # fallback: use stem as-is

print("Paths configured.")
print(f"  Input  : {INPUT_DIR.resolve()}")
print(f"  Output : {OUT_DIR.resolve()}")
print(f"  Markdown: {MD_DIR.resolve()}")

In [ ]:
# ── Cell 2 — Auto-detect inputs, convert PDFs → Markdown ──────────────────────
#
# Accepts any combination of:
#   - PDF files  (auto-converted via pymupdf4llm with page anchors)
#   - Markdown files already converted (used directly)
#
# pymupdf4llm produces better Markdown than raw fitz text extraction:
#   - preserves heading hierarchy (font size → H1/H2/H3)
#   - preserves bold for principle paragraphs
#   - keeps list structure for sub-obligations (a)(b)(c)
#   - inserts page_separator markers we use for page tracking

def pdf_to_markdown(pdf_path: Path, out_path: Path) -> Path:
    """Convert a PDF to Markdown using pymupdf4llm."""
    print(f"  Converting PDF → Markdown: {pdf_path.name}")
    md_text = pymupdf4llm.to_markdown(
        str(pdf_path),
        page_chunks=False,      # single string with \n---\n page separators
        show_progress=False,
    )
    # Inject HTML page anchors matching the format already in pre-converted MDs
    # pymupdf4llm separates pages with  \n-----\n  (5 dashes)
    pages = re.split(r"\n-{3,}\n", md_text)
    annotated = []
    for i, page in enumerate(pages, start=1):
        annotated.append(f'<a id="page-{i}"></a>\n\n## Page {i}\n\n{page.strip()}')
    final_md = "\n\n---\n\n".join(annotated)
    out_path.write_text(final_md, encoding="utf-8")
    print(f"  Saved → {out_path} ({len(pages)} pages)")
    return out_path


def discover_inputs(input_dir: Path, md_dir: Path) -> dict[str, Path]:
    """
    Scan input_dir for .pdf and .md files.
    PDFs are converted to Markdown in md_dir (skipped if already done).
    Returns dict: standard_name → markdown_path
    """
    result = {}

    # Collect all PDFs and MDs
    all_files = list(input_dir.glob("*.pdf")) + list(input_dir.glob("*.md"))
    if not all_files:
        raise FileNotFoundError(
            f"No .pdf or .md files found in {input_dir.resolve()}. "
            "Place your IFRS PDF or Markdown files there and re-run."
        )

    # PDFs that have a corresponding MD in md_dir are not reconverted
    for f in sorted(all_files):
        standard = stem_to_standard(f.stem)
        if f.suffix.lower() == ".pdf":
            md_path = md_dir / f.with_suffix(".md").name
            if not md_path.exists():
                pdf_to_markdown(f, md_path)
            else:
                print(f"  Markdown cache hit: {md_path.name} (skipping conversion)")
            result[standard] = md_path
        elif f.suffix.lower() == ".md":
            # Copy to md_dir so everything is in one place
            md_path = md_dir / f.name
            if not md_path.exists() or md_path.resolve() != f.resolve():
                md_path.write_text(f.read_text(encoding="utf-8"), encoding="utf-8")
            result[standard] = md_path

    return result


print("Discovering input files...")
standard_md_paths = discover_inputs(INPUT_DIR, MD_DIR)

if not standard_md_paths:
    raise RuntimeError("No standards found. Check INPUT_DIR.")

print(f"\nFound {len(standard_md_paths)} standard(s):")
for std, path in standard_md_paths.items():
    print(f"  {std:12s} → {path}")

In [ ]:
# ── Cell 3 — Parse paragraphs from Markdown ───────────────────────────────────
#
# Strategy:
#   1. Read the Markdown line by line.
#   2. Track current page number from <a id="page-N"> anchors.
#   3. Detect section headings (bold/H-level lines) → update current_topic.
#   4. Match paragraph IDs at the start of any line:
#        27        plain number
#        6(a)      sub-paragraph
#        29(a)(i)  nested
#        B62       appendix paragraph
#        B62A      appendix variant
#   5. Accumulate continuation lines until the next paragraph ID.
#   6. Clean up markdown artifacts (##, **, bullet dashes) from body text.
#
# The parser is purely line-based — no page-range assumptions.
# Topic detection is regex-based on the actual section headings in the documents.

# Paragraph ID regex — covers all forms in IFRS S1/S2 including amendments
# Requires 2+ spaces (or tab) after the ID to avoid matching sub-list items
PARA_ID_RE = re.compile(
    r"""^\**                     # optional leading bold markers
    (?P<para_id>
        (?:[A-Z]\d+[A-Z]?)      # appendix: B62, B62A, D4, E1, C1A, C1B
        |
        (?:\d{1,3}               # base number
            (?:\([a-z]{1,4}\)    # (a), (ab)
                (?:\([ivxlcdm]+\) # (i), (ii), (iv)
                    (?:\(\d+\))? # (1), (2)
                )?
            )?
        )
    )\**                         # optional trailing bold markers
    (?:\s{2,}|\t)                # 2+ spaces or tab separator
    (?P<rest>.+)                 # paragraph text
    """,
    re.VERBOSE | re.DOTALL
)

# Section headings — matched against stripped line text (case-insensitive)
# Order matters: more specific patterns first
SECTION_HEADINGS = [
    (re.compile(r"metrics\s+and\s+targets", re.I), "Metrics_targets"),
    (re.compile(r"risk\s+management",       re.I), "Risk_management"),
    (re.compile(r"\bstrategy\b",            re.I), "Strategy"),
    (re.compile(r"\bgovernance\b",          re.I), "Governance"),
    # Appendix sections
    (re.compile(r"application\s+guidance",  re.I), "Appendix_B"),
    (re.compile(r"defined\s+terms",         re.I), "Appendix_A"),
    (re.compile(r"effective\s+date",        re.I), "Appendix_transition"),
]

# Lines to skip — headers, page markers, copyright noise
SKIP_LINE_RE = re.compile(
    r"""^(
        \s*$                          # blank
        |\s*---+\s*$                  # horizontal rule
        |<a\s+id=                     # HTML anchor
        |##\s+Page\s+\d+              # page header injected by converter
        |IFRS\s+S[12]\s+              # running page header
        |RELATED\s+FINANCIAL          # running footer fragment
        |International\s+Sustainability # colophon
        |Columbus\s+Building           # address block
        |Tel\s+\+44                    # phone number
        |ifrs\.org                     # URL
    )""",
    re.VERBOSE | re.IGNORECASE
)

def clean_text(text: str) -> str:
    """Remove markdown formatting artifacts from paragraph body text."""
    text = re.sub(r"\*{1,3}(.+?)\*{1,3}", r"\1", text)   # bold/italic
    text = re.sub(r"^#{1,6}\s+",          "",  text)       # headings
    text = re.sub(r"^[-*]\s+",            "",  text)       # bullet
    text = re.sub(r"\s{2,}",              " ", text)       # extra spaces
    return text.strip()

def is_heading_line(line: str) -> bool:
    """True if the line is a markdown heading (### ...) or a bold-only line."""
    stripped = line.strip()
    if stripped.startswith("#"):
        return True
    # Bold-only line: **text** with nothing else
    if re.match(r"^\*\*[^\*]+\*\*\s*$", stripped):
        return True
    return False

def detect_topic(line: str) -> str | None:
    """Return topic name if this heading line matches a known section."""
    clean = re.sub(r"[#*]", "", line).strip()
    for pattern, topic in SECTION_HEADINGS:
        if pattern.search(clean):
            return topic
    return None


def parse_markdown(md_path: Path, standard: str) -> dict[str, dict]:
    """
    Parse an IFRS Markdown file into a dict of paragraphs.
    Returns: { paragraph_id → { id, standard, topic, text, page, ... } }
    """
    text = md_path.read_text(encoding="utf-8")
    lines = text.splitlines()

    paragraphs     = {}
    current_id     = None
    current_parts  = []
    current_page   = 0
    current_topic  = "Preamble"
    core_started   = False   # flips to True when first numbered para seen

    def flush():
        if current_id is None:
            return
        body = " ".join(clean_text(p) for p in current_parts if clean_text(p))
        body = re.sub(r"\s{2,}", " ", body).strip()
        paragraphs[current_id] = {
            "id":        current_id,
            "standard":  standard,
            "topic":     current_topic,
            "text":      body,
            "page":      current_page,
            "char_count": len(body),
        }

    for line in lines:
        # ── Page number tracking ──────────────────────────────────────────
        page_match = re.search(r'<a\s+id="page-(\d+)"', line)
        if page_match:
            current_page = int(page_match.group(1))
            continue

        # ── Skip noise lines ──────────────────────────────────────────────
        if SKIP_LINE_RE.match(line):
            continue

        # ── Section heading detection ─────────────────────────────────────
        if is_heading_line(line):
            topic = detect_topic(line)
            if topic:
                current_topic = topic
            # Don't flush — headings don't start a new paragraph
            continue

        # ── Paragraph ID detection ────────────────────────────────────────
        m = PARA_ID_RE.match(line.strip())
        if m:
            flush()
            current_id    = m.group("para_id")
            current_parts = [m.group("rest")]
            current_page  = current_page   # carry current page
            core_started  = True
        elif core_started and current_id is not None:
            # Continuation line for the current paragraph
            stripped = line.strip()
            if stripped:
                current_parts.append(stripped)

    flush()   # save the last paragraph

    print(f"  {standard}: {len(paragraphs)} paragraphs parsed")
    return paragraphs


print("Parsing Markdown files...")
all_paragraphs: dict[str, dict[str, dict]] = {}
for standard, md_path in standard_md_paths.items():
    print(f"\n{standard}")
    all_paragraphs[standard] = parse_markdown(md_path, standard)

print(f"\nTotal paragraphs across all standards: "
      f"{sum(len(v) for v in all_paragraphs.values())}")

In [ ]:
# ── Cell 4 — Freeze ifrs_paragraphs.json ─────────────────────────────────────
#
# This file is the immutable source of truth.
# Every downstream validation check references it.
# It is NEVER modified after this cell.

combined = {
    std: list(paras.values())
    for std, paras in all_paragraphs.items()
}

PARA_JSON.write_text(
    json.dumps(combined, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"ifrs_paragraphs.json frozen → {PARA_JSON}")
print("Paragraph counts by standard and topic:")
for std, paras in all_paragraphs.items():
    from collections import Counter
    topics = Counter(p["topic"] for p in paras.values())
    print(f"  {std}:")
    for topic, count in sorted(topics.items()):
        print(f"    {topic:<25} {count}")

In [ ]:
# ── Cell 5 — Dynamic paragraph coverage check ─────────────────────────────────
#
# Instead of a hardcoded list, we:
#   1. Verify all numbered core-content paragraphs form an unbroken sequence
#      (e.g. S2 should have 5, 6, 7 ... 37 with no gaps)
#   2. Verify appendix paragraphs referenced inside the core text actually exist
#   3. Report any gaps clearly
#
# The expected numeric range is auto-detected from the standard's table of
# contents (by finding 'CORE CONTENT' and the next major section after it).

def detect_core_para_range(paragraphs: dict[str, dict]) -> tuple[int, int]:
    """
    Auto-detect the numeric range of core-content paragraphs.
    Looks for paragraphs whose topic is one of the four core areas and
    returns (min_num, max_num) of the numeric IDs found.
    """
    CORE_TOPICS = {"Governance", "Strategy", "Risk_management", "Metrics_targets"}
    nums = []
    for pid, para in paragraphs.items():
        if para["topic"] in CORE_TOPICS:
            m = re.match(r"^(\d+)", pid)
            if m:
                nums.append(int(m.group(1)))
    if not nums:
        return (0, 0)
    return (min(nums), max(nums))


def find_referenced_appendix_ids(paragraphs: dict[str, dict]) -> set[str]:
    """
    Scan all paragraph text for references to appendix paragraphs (B62, B62A, etc.)
    and return the set of referenced IDs.
    """
    APPENDIX_REF_RE = re.compile(
        r"\b([A-Z]\d+[A-Z]?)\b"  # matches B62, B62A, D4, E1 etc.
    )
    refs = set()
    for para in paragraphs.values():
        for m in APPENDIX_REF_RE.finditer(para["text"]):
            refs.add(m.group(1))
    return refs


all_ok = True
coverage_report = []

for standard, paragraphs in all_paragraphs.items():
    available_ids = set(paragraphs.keys())

    # ── Check 1: numeric core range has no gaps ───────────────────────────
    lo, hi = detect_core_para_range(paragraphs)
    if lo and hi:
        expected_nums = set(range(lo, hi + 1))
        found_nums    = {
            int(m.group(1))
            for pid in available_ids
            if (m := re.match(r"^(\d+)$", pid))
        }
        missing_nums = expected_nums - found_nums
        if missing_nums:
            all_ok = False
            coverage_report.append(
                f"  {standard}: Missing numeric paragraphs in range "
                f"{lo}–{hi}: {sorted(missing_nums)}"
            )
        else:
            coverage_report.append(
                f"  {standard}: Numeric core range {lo}–{hi} complete ✓ "
                f"({hi - lo + 1} paragraphs)"
            )
    else:
        coverage_report.append(f"  {standard}: WARNING — no core topic paragraphs detected")
        all_ok = False

    # ── Check 2: appendix IDs referenced in text exist ────────────────────
    referenced = find_referenced_appendix_ids(paragraphs)
    # Filter to appendix IDs only (start with a letter)
    appendix_refs = {r for r in referenced if re.match(r"^[A-Z]\d+", r)}
    missing_appendix = appendix_refs - available_ids
    if missing_appendix:
        coverage_report.append(
            f"  {standard}: Appendix IDs referenced but not found: "
            f"{sorted(missing_appendix)}"
        )
        # This is a warning, not a hard failure — appendix may be in a separate volume
    else:
        coverage_report.append(
            f"  {standard}: All {len(appendix_refs)} referenced appendix IDs found ✓"
        )

print("Coverage check:")
for line in coverage_report:
    print(line)

if not all_ok:
    print(
        "\n⚠  Some paragraphs are missing. "
        "Check the parser output above and inspect the Markdown source."
        "\nFix the parser before running the LLM cells."
    )
else:
    print("\n✓ All coverage checks passed.")

In [ ]:
# ── Cell 6 — Classify requirement paragraphs (rule-based, no LLM) ─────────────
#
# A paragraph is a disclosure requirement if it contains a positive SHALL
# obligation — i.e. "shall disclose", "shall provide", "shall describe", etc.
#
# We apply this across ALL paragraphs (core + appendix) because:
#   - Appendix B of S2 contains SHALL requirements (B62, B63, etc.)
#   - Appendix D of S1 contains some requirements
# The topic tag already tells us which section they belong to.

POSITIVE_OBLIGATION_RE = re.compile(
    r"\bshall\s+(?:"
    r"disclose\b|provide\b|include\b|describe\b|explain\b"
    r"|identify\b|present\b|report\b|state\b|specify\b"
    r"|apply\b|use\b|consider\b|assess\b|measure\b"
    r")",
    re.IGNORECASE
)

OBJECTIVE_RE = re.compile(
    r"^(?:the\s+)?objective\s+of", re.IGNORECASE
)

def is_requirement(para: dict) -> bool:
    text = para["text"]
    text_lower = text.lower()
    if "shall" not in text_lower:
        return False
    if OBJECTIVE_RE.match(text_lower):
        return False
    return bool(POSITIVE_OBLIGATION_RE.search(text))


requirement_paragraphs: dict[str, dict[str, dict]] = {}

for standard, paragraphs in all_paragraphs.items():
    requirement_paragraphs[standard] = {
        pid: para
        for pid, para in paragraphs.items()
        if is_requirement(para)
    }
    total = len(paragraphs)
    reqs  = len(requirement_paragraphs[standard])
    print(f"{standard}: {reqs} requirement paragraphs out of {total} total")
    print(f"  IDs: {sorted(requirement_paragraphs[standard].keys(), key=lambda x: (len(x), x))}")

In [ ]:
# ── Cell 7 — LLM structuring ──────────────────────────────────────────────────
#
# The LLM receives ONLY the verified paragraph text from Cell 3.
# It cannot reference paragraphs that do not exist in all_paragraphs.
# It returns structured JSON; every row is validated in Cell 8.
#
# The system prompt is deliberately strict:
#   - source_quote must be a verbatim substring of the input text
#   - requirement_summary must paraphrase, not add new requirements
#   - one row per sub-obligation (a), (b), (c)

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from environment

SYSTEM_PROMPT = """\
You are an IFRS disclosure analyst extracting structured requirement data.

You will receive the text of ONE IFRS paragraph. Extract its disclosure structure.

STRICT RULES — violating any of these invalidates your output:
1. source_quote MUST be a verbatim substring of the paragraph text provided.
   Copy it character-for-character. Max 300 characters. It must contain the core
   obligation verb (shall disclose / shall provide / shall describe / etc).
2. requirement_summary: one sentence paraphrase of what the entity must do.
   Do NOT add requirements not stated in the text.
3. required_evidence: list only items explicitly mentioned in the paragraph.
   Do not infer or extend beyond the text.
4. If the paragraph has multiple sub-obligations (a), (b), (c) at the same
   nesting level, return ONE row per sub-obligation.
   Use paragraph_ref like "6(a)", "6(b)" for each.
   If the paragraph is a single obligation, return one row with just "6".
5. critical = true only when the paragraph uses "shall disclose" directly
   (not "shall consider" or auxiliary phrases).
6. Return ONLY a valid JSON array. No explanation. No markdown fences.

Each element in the array:
{
  "paragraph_ref":      string,   // e.g. "6(a)" or "29"
  "requirement_summary": string,  // one sentence
  "source_quote":        string,  // verbatim substring ≤300 chars
  "required_evidence":   [string],// explicit items from text only
  "disclosure_type":     "narrative" | "quantitative" | "mixed",
  "critical":            boolean
}
"""


def structure_paragraph(
    standard: str, para_id: str, para: dict
) -> list[dict]:
    """
    Send one paragraph to the LLM and return a list of structured requirement rows.
    Returns [] on parse failure (row will be recorded as invalid).
    """
    user_msg = (
        f"Standard: {standard}\n"
        f"Paragraph ID: {para_id}\n"
        f"Topic section: {para['topic']}\n"
        f"Page: {para['page']}\n\n"
        f"Paragraph text:\n{para['text']}"
    )

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = response.content[0].text.strip()
    except Exception as e:
        print(f"  API error for {standard}/{para_id}: {e}")
        return []

    # Strip markdown fences if present
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$",          "", raw)

    try:
        rows = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  JSON parse error for {standard}/{para_id}: {e}")
        return []

    if isinstance(rows, dict):
        rows = [rows]
    if not isinstance(rows, list):
        return []

    # Tag every row with provenance
    for row in rows:
        row["standard"]             = standard
        row["topic"]                = para["topic"]
        row["source_paragraph_ids"] = [para_id]
        row["source_page"]          = para["page"]
        row["source_text_hash"]     = hashlib.sha256(
            para["text"].encode()
        ).hexdigest()[:16]

    return rows


# ── Run LLM structuring across all requirement paragraphs ─────────────────────
raw_catalog_rows: list[dict] = []

for standard, paras in requirement_paragraphs.items():
    print(f"\nStructuring {standard} — {len(paras)} paragraphs...")
    for para_id, para in tqdm(paras.items(), desc=standard):
        rows = structure_paragraph(standard, para_id, para)
        raw_catalog_rows.extend(rows)

print(f"\nRaw rows before validation: {len(raw_catalog_rows)}")

In [ ]:
# ── Cell 8 — Validate every row (grounding check) ────────────────────────────
#
# A row is valid only if its source_quote is a verifiable substring
# of the extracted paragraph text in all_paragraphs.
#
# Fuzzy fallback: if exact match fails, check 80% word overlap.
# Rows failing both are rejected — not relaxed.

def validate_row(row: dict) -> tuple[bool, str]:
    if not row.get("paragraph_ref"):
        return False, "Missing paragraph_ref"

    quote = row.get("source_quote", "").strip()
    if len(quote) < 20:
        return False, f"source_quote too short ({len(quote)} chars): '{quote}'"

    standard   = row.get("standard", "")
    source_ids = row.get("source_paragraph_ids", [])
    quote_lower = quote.lower()

    for pid in source_ids:
        para = all_paragraphs.get(standard, {}).get(pid, {})
        if not para:
            continue
        para_text = para.get("text", "").lower()

        # Exact substring match
        if quote_lower in para_text:
            return True, "ok"

        # Fuzzy: word overlap ≥ 80%
        q_words = set(re.findall(r"\b\w+\b", quote_lower))
        p_words = set(re.findall(r"\b\w+\b", para_text))
        if q_words:
            overlap = len(q_words & p_words) / len(q_words)
            if overlap >= 0.80:
                row["grounding_note"] = f"fuzzy {overlap:.0%} word overlap"
                return True, "ok (fuzzy)"

    if not source_ids:
        return False, "source_paragraph_ids is empty"

    return False, (
        f"source_quote not found in source paragraph text. "
        f"Quote: '{quote[:80]}'"
    )


valid_rows:   list[dict] = []
invalid_rows: list[dict] = []

for row in raw_catalog_rows:
    # Also check required_evidence and requirement_summary
    if not row.get("required_evidence"):
        row["rejection_reason"] = "required_evidence is empty"
        invalid_rows.append(row)
        continue
    if not row.get("requirement_summary", "").strip():
        row["rejection_reason"] = "requirement_summary is empty"
        invalid_rows.append(row)
        continue

    ok, reason = validate_row(row)
    if ok:
        valid_rows.append(row)
    else:
        row["rejection_reason"] = reason
        invalid_rows.append(row)

print(f"Valid rows   : {len(valid_rows)}")
print(f"Rejected rows: {len(invalid_rows)}")
if invalid_rows:
    print("\nRejection details:")
    for r in invalid_rows:
        print(f"  {r.get('standard')}/{r.get('paragraph_ref')}: "
              f"{r['rejection_reason']}")

In [ ]:
# ── Cell 9 — Finalize catalog ─────────────────────────────────────────────────
#
# Assign disclosure_id, set applies_when, deduplicate, save CSV + JSON.
#
# applies_when logic:
#   IFRS_S2 rows        → climate_only
#   IFRS_S1 core areas  → general_sustainability  (Governance/Strategy/Risk/Metrics)
#   IFRS_S1 other       → always

CORE_AREA_TOPICS = {"Governance", "Strategy", "Risk_management", "Metrics_targets"}

def make_disclosure_id(standard: str, para_ref: str) -> str:
    """S2 + 6(a) → S2.6a | S1 + 27 → S1.27 | S2 + B62 → S2.B62"""
    prefix = "S1" if "S1" in standard else "S2"
    clean  = re.sub(r"[()]", "", para_ref).replace(" ", "_")
    return f"{prefix}.{clean}"


def applies_when(row: dict) -> str:
    if "S2" in row.get("standard", ""):
        return "climate_only"
    if row.get("topic", "") in CORE_AREA_TOPICS:
        return "general_sustainability"
    return "always"


seen_ids: dict[str, int] = {}
finalized: list[dict] = []

for row in valid_rows:
    base_id = make_disclosure_id(row["standard"], row["paragraph_ref"])

    if base_id in seen_ids:
        seen_ids[base_id] += 1
        disc_id = f"{base_id}_{seen_ids[base_id]}"
    else:
        seen_ids[base_id] = 1
        disc_id = base_id

    finalized.append({
        "disclosure_id":        disc_id,
        "standard":             row["standard"],
        "core_area":            row.get("topic", "Unknown"),
        "paragraph_ref":        row["paragraph_ref"],
        "source_paragraph_ids": json.dumps(row.get("source_paragraph_ids", [])),
        "source_quote":         row.get("source_quote", ""),
        "requirement_summary":  row.get("requirement_summary", ""),
        "required_evidence":    json.dumps(row.get("required_evidence", [])),
        "disclosure_type":      row.get("disclosure_type", "narrative"),
        "critical":             row.get("critical", False),
        "applies_when":         applies_when(row),
        "source_page":          row.get("source_page"),
        "source_text_hash":     row.get("source_text_hash", ""),
        "grounding_note":       row.get("grounding_note", ""),
    })

df = pd.DataFrame(finalized)

# Integrity assertions
assert df["disclosure_id"].duplicated().sum() == 0,         "Duplicate disclosure_ids found"
assert df["source_quote"].notna().all(),                     "Null source_quotes found"
assert (df["source_quote"].str.len() >= 20).all(),           "source_quotes too short"
assert not df["requirement_summary"].str.strip().eq("").any(), "Empty requirement_summaries"

# Save CSV
df.to_csv(CATALOG_CSV, index=False, encoding="utf-8")

# Save JSON with arrays parsed back from JSON strings
json_rows = []
for _, row in df.iterrows():
    r = row.to_dict()
    r["source_paragraph_ids"] = json.loads(r["source_paragraph_ids"])
    r["required_evidence"]    = json.loads(r["required_evidence"])
    json_rows.append(r)

CATALOG_JSON.write_text(
    json.dumps(json_rows, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Catalog saved: {len(df)} rows")
print(f"  CSV  → {CATALOG_CSV}")
print(f"  JSON → {CATALOG_JSON}")
print("\nBreakdown by standard × topic:")
print(df.groupby(["standard", "core_area"]).size().to_string())

In [ ]:
# ── Cell 10 — Write validation_report.md ──────────────────────────────────────

exact_matches = sum(1 for r in finalized if not r.get("grounding_note"))
fuzzy_matches = sum(1 for r in finalized if r.get("grounding_note"))

lines = [
    "# IFRS Disclosure Catalog — Validation Report",
    f"\nGenerated: {datetime.now().isoformat()}",
    "\n## Input files",
]
for std, path in standard_md_paths.items():
    lines.append(f"- `{std}` → `{path.name}`")

lines += [
    "\n## Paragraph parsing",
]
for std, paras in all_paragraphs.items():
    from collections import Counter
    topics = Counter(p["topic"] for p in paras.values())
    lines.append(f"\n### {std}")
    lines.append(f"Total paragraphs: {len(paras)}")
    for topic, count in sorted(topics.items()):
        lines.append(f"- {topic}: {count}")

lines += [
    "\n## Coverage checks",
    *coverage_report,

    "\n## Requirement classification (SHALL filter)",
]
for std, paras in requirement_paragraphs.items():
    lines.append(f"- {std}: {len(paras)} requirement paragraphs")

lines += [
    "\n## LLM structuring",
    f"- Raw rows produced: {len(raw_catalog_rows)}",
    f"- Valid rows: {len(valid_rows)}",
    f"- Rejected rows: {len(invalid_rows)}",

    "\n## Grounding verification",
    "Every source_quote was verified as a substring of its source paragraph text.",
    f"- Exact matches: {exact_matches}",
    f"- Fuzzy matches (≥80% word overlap): {fuzzy_matches}",

    "\n## Rejected rows",
]
if invalid_rows:
    for r in invalid_rows:
        lines.append(
            f"- `{r.get('standard')}/{r.get('paragraph_ref')}`: "
            f"{r.get('rejection_reason')}"
        )
else:
    lines.append("- None. All rows passed validation.")

lines += [
    "\n## Final catalog breakdown",
    "```",
    df.groupby(["standard", "core_area"]).size().to_string(),
    "```",

    "\n## Output files",
    f"- `{PARA_JSON.name}` — raw extracted paragraphs (immutable source of truth)",
    f"- `{CATALOG_CSV.name}` — disclosure catalog (CSV)",
    f"- `{CATALOG_JSON.name}` — disclosure catalog (JSON)",
    f"- `{VAL_REPORT.name}` — this report",

    "\n## Key invariant",
    "Every `source_quote` in the catalog is traceable to a specific page and",
    "paragraph in `ifrs_paragraphs.json`. The LLM never invented references.",
]

VAL_REPORT.write_text("\n".join(lines), encoding="utf-8")
print(f"validation_report.md → {VAL_REPORT}")

In [ ]:
# ── Cell 11 — Embed into ChromaDB (run after catalog is stable) ───────────────
#
# Run this cell only after you are satisfied with the catalog in Cell 9.
# It is safe to re-run — collection.upsert is idempotent.
#
# The embed text is requirement_summary + source_quote combined,
# giving the vector store both the meaning and the verbatim obligation.

try:
    import chromadb
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--break-system-packages",
        "chromadb", "sentence-transformers"
    ])
    import chromadb
    from sentence_transformers import SentenceTransformer


EMBED_MODEL_NAME = "all-MiniLM-L6-v2"   # fast + good retrieval quality
CHROMA_PATH      = OUT_DIR / "chroma_db"

print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection    = chroma_client.get_or_create_collection(
    name="ifrs_disclosures",
    metadata={"hnsw:space": "cosine"},
)

documents  = []
metadatas  = []
ids        = []

for _, row in df.iterrows():
    embed_text = f"{row['requirement_summary']} {row['source_quote']}"
    documents.append(embed_text)
    metadatas.append({
        "disclosure_id":   row["disclosure_id"],
        "standard":        row["standard"],
        "core_area":       row["core_area"],
        "paragraph_ref":   row["paragraph_ref"],
        "critical":        str(row["critical"]),
        "applies_when":    row["applies_when"],
        "disclosure_type": row["disclosure_type"],
    })
    ids.append(row["disclosure_id"])

print(f"Encoding {len(documents)} disclosure rows...")
embeddings = embed_model.encode(documents, show_progress_bar=True).tolist()

collection.upsert(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids,
)

print(f"\nChromaDB collection count: {collection.count()}")
print(f"Persisted → {CHROMA_PATH}")

In [ ]:
# ── Cell 12 — Judge agent retrieval pattern ───────────────────────────────────
#
# This cell shows how the downstream judge agent queries the catalog.
# It is reference code — not part of the catalog build pipeline.

def get_relevant_disclosures(
    report_section_text: str,
    core_area: str | None = None,
    standard_filter: str | None = None,
    applies_when_filter: str | None = None,
    top_k: int = 10,
) -> list[dict]:
    """
    Retrieve the most relevant disclosure requirements for a report section.

    Args:
        report_section_text:  The text of the section being evaluated.
        core_area:            Filter to one topic ("Governance", "Strategy", etc).
        standard_filter:      "IFRS_S1" or "IFRS_S2" or None for both.
        applies_when_filter:  "climate_only", "general_sustainability", "always", or None.
        top_k:                Number of results to return.
    """
    where_clauses = []
    if core_area:
        where_clauses.append({"core_area": {"$eq": core_area}})
    if standard_filter:
        where_clauses.append({"standard": {"$eq": standard_filter}})
    if applies_when_filter:
        where_clauses.append({"applies_when": {"$eq": applies_when_filter}})

    where = None
    if len(where_clauses) == 1:
        where = where_clauses[0]
    elif len(where_clauses) > 1:
        where = {"$and": where_clauses}

    query_vec = embed_model.encode([report_section_text]).tolist()

    kwargs = dict(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    return [
        {
            "disclosure_id":    doc_id,
            "metadata":         results["metadatas"][0][i],
            "requirement_text": results["documents"][0][i],
            "similarity":       round(1 - results["distances"][0][i], 4),
        }
        for i, doc_id in enumerate(results["ids"][0])
    ]


def build_judge_prompt(report_section: str, core_area: str) -> str:
    disclosures = get_relevant_disclosures(report_section, core_area=core_area)

    req_block = "\n".join(
        f"[{d['disclosure_id']}] {d['requirement_text']}"
        for d in disclosures
    )

    return f"""You are an IFRS S1/S2 compliance judge.

REQUIREMENTS (sourced from verified IFRS paragraphs):
{req_block}

REPORT SECTION TO EVALUATE:
{report_section}

For each requirement ID above, return a JSON array:
{{
  "disclosure_id": "...",
  "verdict": "PASS" | "PARTIAL" | "FAIL",
  "evidence_found": "quoted sentence from report, or null",
  "gap": "what is missing, or null if PASS"
}}
"""


# ── Quick smoke test ─────────────────────────────────────────────────────────
test_section = """
The Board of Directors oversees climate-related risks through its Risk Committee,
which meets quarterly to review climate scenarios and transition plans.
The Chief Sustainability Officer reports directly to the Board.
"""

results = get_relevant_disclosures(
    test_section,
    core_area="Governance",
    top_k=5,
)

print(f"Top {len(results)} governance disclosures for test section:")
for r in results:
    print(f"  [{r['disclosure_id']}] sim={r['similarity']:.3f}  "
          f"{r['requirement_text'][:80]}...")